## Content-based filtering

In this document, we implement a content-based filtering recommender system that suggests movies by analyzing their attributes and comparing them to a user's profile. Unlike collaborative filtering, which relies on user interactions and preferences across a broad audience, content-based filtering focuses solely on the characteristics of the movies themselves.

Our system builds a user profile based on the movies they have liked or interacted with in the past. We extract key attributes such as genres, keywords, tagline, and overview from the dataset and use them to find similarities between the user’s preferences and other movies. By leveraging text similarity techniques and genre-matching, the model recommends movies that closely align with the user's taste.

To evaluate the effectiveness of our recommendations, we apply various performance metrics like Precision, Recall, F1-score, and Text Similarity Score. These metrics help us assess how well the system captures user preferences and provides relevant movie suggestions.

### Libraries

In [1]:
import numpy as np
import pandas as pd
import sys
import os
from pathlib import Path
import pandas as pd
import ast
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle
from typing import List

PROJECT_ROOT = Path(os.getcwd()).parent
sys.path.append(str(PROJECT_ROOT))
import isa_project_1.config as config
from testing import recommend_movies, test_accuracy

2025-03-26 22:59:50.935 | INFO     | ML_pipeline.isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ML_pipeline


2025-03-26 22:59:50.971 | INFO     | isa_project_1.config:<module>:11 - PROJ_ROOT path is: C:\Users\fmojt\Documents\DataAnalysisProjects\Mini-project1_ISA\deployment\ML_pipeline


### Loading dataset

In [2]:
print("opening dataset...")
# here we read the respective output file where the dataset was stored 
movie_df = pd.read_csv(config.INTERIM_DATASETS['tmdb_5000_movies.csv'])
print("success!")

opening dataset...
success!


### Data Processing

First the data must be properly preprocessed. This includes following actions:

1) **Replacing NaNs**
2) **Extracting key data from JSON-like objects**
3) **Concentating the independent variables (predictors)**
4) **Futher text preprocessing (to lowercase, removing non-character letters, removing stopwords)**

#### nltk library

nltk is a useful python library that work with human language data. In the content of our project, it's going to help by **providing stopwords** that are available in English language.

In [3]:
# downloading stopwords (only needed once)
nltk.download('stopwords')
# here we use the unordered set (represented by hash table) to optimize stopword search
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\fmojt\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


#### Replacing NaN values

In [4]:
# filling NaN values with empty strings
movie_df['overview'] = movie_df['overview'].fillna('')
movie_df['tagline'] = movie_df['tagline'].fillna('')

#### Extracting key attributes from JSON-like strings

Attributes *genres* and *keywords* which make up our predictors are stored in JSON-like lists. We'are going to solve this issue by extracting only the **name key** and replacing the entire list with it.

For this purpose, we have declared a separate function and defined it below.

In [5]:
# Function to extract text from JSON-like columns
def extract_names(text: str, key: str):
    """Converts the provided JSON-like list into a string of one its keys delimited by whitespace.

    Args:
        text (str): the JSON list to be converted
        key (str): the element key to extract

    Returns:
        str: final string of extracted keys
    """
    try:
        # convert the string dictionary into python dictionary
        lst = ast.literal_eval(text)

        # extracting the key
        return ' '.join([i[key] for i in lst])
    except (ValueError, SyntaxError):
        return ''

Now to replace the strings we simply call the function.

In [6]:
# Apply extraction function to genres and keywords
movie_df['genres'] = movie_df['genres'].apply(extract_names, key='name')
movie_df['keywords'] = movie_df['keywords'].apply(extract_names, key='name')

#### Concentatining the independent variables

It's a good practice to combine all text predictors into single text by creating a separate feature *combined_text*. 

In [7]:
# combinining relevant features into single text
movie_df['combined_text'] = movie_df['overview'] + ' ' + movie_df['tagline'] + ' ' + movie_df['genres'] + ' ' + movie_df['keywords']

#### Further text preprocessing

Finally the combined text needs some additional preprocessing:
1) making the text lowercase
2) removing non-word characters (!@#$%^&, etc.)
3) removing stopwords

In [8]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text: str):
    """Converts text to lowercase, removes special characters and stopwords, and applies lemmatization."""
    
    text = text.lower()
    text = re.sub(r'\W+', ' ', text)  # Remove special characters
    words = text.split()
    
    # Remove stopwords and lemmatize
    text = ' '.join([lemmatizer.lemmatize(word) for word in words if word not in stop_words])
    
    return text

movie_df['processed_text'] = movie_df['combined_text'].apply(preprocess_text)

### Feature extraction method - TF-IDF

In the data-preprocessing we have created a single attribute which combines all of the predictors. These predictors had been preprocessed to remove any unwanted phenomenom. Now it is time to create a **TF-IDF matrix**.

We use the **TfidfVectorizer** from *sklearn* to vectorize all the movies. By doing so we'll get a TF_IDF matrix containing documents (rows) and all words (columns). Every value represent measure of how important a word is to the entire corpus.

In [9]:
# TF-IDF Vectorization
# TF-IDF is a statistical measure that evaluates how important a word (term) is
# to a document or collection (corpus)

# TF-IDF = TF x IDF
# TF = occur_of_term_in_document / total_words_in_document
# IDF = total_num_of_documents / num_of_documents_containing_the_term
tfidf_vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf_vectorizer.fit_transform(movie_df['processed_text'])

#Initialize CountVectorizer with stop words
count_vectorizer = CountVectorizer(stop_words='english')

# Apply CountVectorizer on the processed text column
count_matrix = count_vectorizer.fit_transform(movie_df['processed_text'])

### Testing

After the data is successfully preprocessed, we can start testing. The first feature defined by our business goal is the user profile. In our context, the user profile is a complete history of movies that they liked or interacted with in some way.

For development purposes, we've decided to use a list of n randomly selected movies from the dataset. Note that n is represented by the USER_LIKED_MOVIES constant.

Now, in order to test the dataset, we're going to use various metrics like Precision, Recall, or F1-score. We can't use Accuracy here because it is tricky to define TN (genres that were not recommended and not liked). Since this number is enormous, TN becomes less meaningful. Instead, we define a simplified alternative:

*hit_rate = TP / TotalLikedGenres*

In the context of our model, the metrics mean the following:

1) **Precision** - how many recommended genres actually match the user’s liked genres, compared to the total number of recommended genres?

2) **Recall** - how many of the user's liked genres were successfully recommended, compared to the total liked genres?

3) **F1-score** - the harmonic mean between Precision and Recall.

4) **Text Similarity Score** - measures how similar the textual features (e.g., movie overviews, keywords, or taglines) of the recommended movies are to those of the user's liked movies. This is based on techniques like CountVectorizer, TF-IDF, or word embeddings.

By using these metrics, we evaluate how well the recommendation system aligns with the user's preferences, both in terms of genre and textual characteristics.

During testing we are going to utilize several functions defined below.

In [10]:
# def recommend_movies(user_liked_movies: List[str], matrix, count: int = 10):
#     # Randomly select n movies from the entire dataset
#     user_liked_movies = movie_df.sample(n=len(user_liked_movies), random_state=42)['title'].tolist()
#     user_liked_movies = pd.Series(user_liked_movies)

#     # Get all movie vectors for movies the user liked
#     liked_movie_indices = movie_df[movie_df['title'].isin(user_liked_movies)].index
#     # we get the subset of the entire tfidf matrix based on user liked movies
#     liked_movie_vectors = matrix[liked_movie_indices]

#     # mean is calculated of all movies user has interacted with
#     # the result needs to be converted into 2D array with one row (1 x N)
#     user_profile_vector = np.asarray(liked_movie_vectors.mean(axis=0)).reshape(1, -1)

#     user_similarities = cosine_similarity(user_profile_vector, matrix).flatten()

#     # Create DataFrame of movies with similarity scores
#     movie_scores = pd.DataFrame({'title': movie_df['title'], 'similarity': user_similarities})

#     # Filter out movies the user has already interacted with
#     movie_scores = movie_scores[~movie_scores['title'].isin(user_liked_movies)]

#     # Return top recommendations after filtering
#     return movie_scores.sort_values(by='similarity', ascending=False).head(count)

# def get_movie_texts(movie_titles):
#     """Retrieve processed_text for given movie titles."""
#     return movie_df[movie_df['title'].isin(movie_titles)]['processed_text'].tolist()

# def test_accuracy(recommended_movies: pd.DataFrame, user_liked_movies: List[str], vectorizer: TfidfVectorizer | CountVectorizer, USER_LIKED_MOVIES_COUNT: int = 100):
#     recommended_movies_list = list(recommended_movies['title'])
    
#     # Get genres
#     user_genres = get_movie_genres(user_liked_movies)
#     recommended_genres = get_movie_genres(recommended_movies_list)

#     # Calculate genre overlap
#     TP_genre = len(user_genres & recommended_genres)  # Common genres
#     FP_genre = len(recommended_genres - user_genres)  # Mismatched genres
#     FN_genre = len(user_genres - recommended_genres)  # Missed genres

#     # Calculate precision, recall, and F1-score based on genres
#     precision_genre = TP_genre / (TP_genre + FP_genre) if (TP_genre + FP_genre) > 0 else 0
#     recall_genre = TP_genre / (TP_genre + FN_genre) if (TP_genre + FN_genre) > 0 else 0
#     f1_genre = (2 * precision_genre * recall_genre) / (precision_genre + recall_genre) if (precision_genre + recall_genre) > 0 else 0

#     #Get processed_text of movies
#     user_texts = get_movie_texts(user_liked_movies)
#     recommended_texts = get_movie_texts(recommended_movies_list)

#     # TF-IDF vectorization
#     # vectorizer = TfidfVectorizer(stop_words='english')
#     tfidf_matrix = vectorizer.fit_transform(user_texts + recommended_texts)

#     # Compute cosine similarity between liked & recommended movies
#     user_vectors = tfidf_matrix[:len(user_texts)]
#     recommended_vectors = tfidf_matrix[len(user_texts):]

#     similarity_matrix = cosine_similarity(user_vectors, recommended_vectors)
#     avg_similarity = np.mean(similarity_matrix)  # Average similarity across all pairs

#     return {
#         "Genre Precision": f"{precision_genre:.4f}",
#         "Genre Recall": f"{recall_genre:.4f}",
#         "Genre F1-score": f"{f1_genre:.4f}",
#         "Text Similarity Score": f"{avg_similarity:.4f}"
#     }    

# def get_movie_genres(movie_titles):
#     genres = movie_df[movie_df['title'].isin(movie_titles)]['genres']
#     return set(genre.strip() for genre_list in genres.dropna() for genre in genre_list.split())

### Scenario 1 - user_liked_movies > recommended_movies

First we are going to test the more regular case occuring in recommender system. User profile consists of movies user has already interacted with. As user interacts with the system this number grows and is generally greater than the amount of recommended movies in a single session.

In [11]:
USER_LIKED_MOVIES_COUNT = 100
RECOMMENDED_MOVIES = 10

#### Recommending movies

In [12]:
# user_profile = create_user_profile_vector(USER_LIKED_MOVIES_COUNT)
# Randomly select n movies from the entire dataset
user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()
recommended_movies = recommend_movies(movie_df=movie_df, count=RECOMMENDED_MOVIES, user_liked_movies=user_liked_movies, matrix=count_matrix)

# print("Interacted movies:", user_liked_movies)

print("Top Recommended Movies:")

recommended_movies_list = list(recommended_movies['title'])
recommended_movies_list

Top Recommended Movies:


['Four Single Fathers',
 'Boynton Beach Club',
 'The Last Song',
 'Solitary Man',
 "Jesus' Son",
 'The Incredibly True Adventure of Two Girls In Love',
 'When Harry Met Sally...',
 'About Time',
 'Life During Wartime',
 'Raising Helen']

#### Testing model's accuracy

We utilize the test_accuracy function and we calculate the model's accuracy using various metrics. 

In [13]:
accuracies = test_accuracy(movie_df=movie_df, recommended_movies=recommended_movies, vectorizer=count_vectorizer, user_liked_movies=user_liked_movies)
accuracies

{'Genre Precision': '1.0000',
 'Genre Recall': '0.3158',
 'Genre F1-score': '0.4800',
 'Text Similarity Score': '0.0879'}

### Scenario 1 - user_liked_movies < recommended_movies

This scenario is more unlikely to happen. However we still test how it influence the metrics.

In [14]:
USER_LIKED_MOVIES_COUNT = 10
RECOMMENDED_MOVIES = 100

#### Recommending movies

In [15]:
# user_profile = create_user_profile_vector(USER_LIKED_MOVIES_COUNT)
# Randomly select n movies from the entire dataset
user_liked_movies = movie_df.sample(n=USER_LIKED_MOVIES_COUNT, random_state=42)['title'].tolist()
recommended_movies = recommend_movies(movie_df=movie_df, count=RECOMMENDED_MOVIES, user_liked_movies=user_liked_movies, matrix=count_matrix)

# print("Interacted movies:", user_liked_movies)

print("Top Recommended Movies:")

recommended_movies_list = list(recommended_movies['title'])
recommended_movies_list

Top Recommended Movies:


['The Helix... Loaded',
 'Code 46',
 'Trash',
 'Proof of Life',
 'Four Single Fathers',
 'Hav Plenty',
 'Journey to the Center of the Earth',
 '3 Days to Kill',
 'In Too Deep',
 'Brooklyn Rules',
 'Echo Dr.',
 'Silver Medalist',
 'Spy Kids',
 'Jimmy Neutron: Boy Genius',
 'Meet the Fockers',
 'The Host',
 'Flatliners',
 'The Croods',
 'About Time',
 'Megiddo: The Omega Code 2',
 'Mission: Impossible',
 'Dragon Wars: D-War',
 'Ask Me Anything',
 'Surrogates',
 'Aliens in the Attic',
 'Crying with Laughter',
 'Vanilla Sky',
 'Children of Men',
 'Æon Flux',
 'Impostor',
 'I Am Number Four',
 'Mad Max: Fury Road',
 'Southland Tales',
 'The Long Kiss Goodnight',
 'Terminator Genisys',
 'I Am Legend',
 'The Quiet American',
 'The Last Song',
 'The Pacifier',
 'Lonesome Jim',
 '9',
 'The Brothers Bloom',
 'Mission: Impossible III',
 'Flightplan',
 'G.I. Joe: The Rise of Cobra',
 'An Education',
 'Zipper',
 'Spy Kids 3-D: Game Over',
 'Earth to Echo',
 'Mission: Impossible II',
 'Frequency',
 

#### Testing model's accuracy

In [16]:
accuracies = test_accuracy(movie_df=movie_df, recommended_movies=recommended_movies, vectorizer=count_vectorizer, user_liked_movies=user_liked_movies)
accuracies

{'Genre Precision': '0.7647',
 'Genre Recall': '1.0000',
 'Genre F1-score': '0.8667',
 'Text Similarity Score': '0.0714'}

### Saving TF-IDF matrix&vectorizer

Finally, we save the both the TF-IDF matrix and vectorizer to .pkl files (binary format).

In [17]:
# Save the TF-IDF matrix and vectorizer
with open(config.MODELS_DIR / "tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open(config.MODELS_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf_vectorizer, f)

with open(config.MODELS_DIR / "count_matrix.pkl", "wb") as f:
    pickle.dump(count_matrix, f)

with open(config.MODELS_DIR / "count_vectorizer.pkl", "wb") as f:
    pickle.dump(count_vectorizer, f)

movie_df.to_pickle(config.PROCESSED_DATA_DIR / "movie_df.pkl")

### Conclusion

In this document, we have successfully preprocessed predictors and the target variable. We then created a user profile of n randomly-selected interacted movies, calculated the similarity between the user profile and other movies based on attributes like overview, tagline, keywords, and genres, and recommended the top k most similar movies. Lastly, we tested the predictions using various metrics.

#### Model's Evaluation

We evaluated the model using **Precision**, **Recall**, and **F1-score** based on genres instead of direct movie matches. We also incorporated a Text Similarity Score to measure how well the recommended movies match the user's profile based on textual attributes. We did not apply traditional accuracy due to the enormous amount of non-liked movies, which makes defining True Negatives less meaningful.

During testing, we analyzed two possible scenarios:

1) recommended_movies < liked_movies

2) recommended_movies > liked_movies

Based on the metric outcomes, we can deduce several things:

In the first case, precision **is usually higher and recall lower**. This is because the model primarily recommends movies that align closely with the user's genre preferences, avoiding mismatched genres. However, if the number of recommended movies is significantly lower than the user's profile size, it is logical that the model fails to cover all genres the user interacts with. This scenario is more natural in a recommender system, where fewer but highly relevant recommendations are preferred.

The second case is the exact opposite. **Recall is higher because many user-liked genres appear in recommendations, but precision is lower because new, unknown genres may also be recommended** based on similarity. This trade-off highlights the balance between recommending relevant genres and exploring new potential interests.

Overall, incorporating both genre-based evaluation and text similarity helps provide a more comprehensive assessment of the model's performance.

Finally, the system was stored as **.pkl file** and is now ready for deployment.